# DDL: `dbspend360_pool_dbu_cost`

Creates the per-pool / per-cluster / per-day Databricks DBU cost table populated by `dbspend360_pool_dbu_cost_app`.

Sibling of `dbspend360_dbu_cost` and `dbspend360_all_purpose_dbu_cost`, but keyed on `(instance_pool_id, cluster_id, usage_date)`.
Filters `system.billing.usage` to `usage_metadata.instance_pool_id IS NOT NULL` (no `cluster_source` filter — pool-backed
clusters of any source contribute). Rows where `usage_metadata.cluster_id` is NULL bucket into `cluster_id = '__pool_overhead__'`
rather than being dropped, so pool-level bootstrap charges stay accounted for.

**Widgets**
- `catalog` - target Unity Catalog name
- `schema`  - target schema name within `catalog`

In [ ]:
dbutils.widgets.text("catalog", "", "Catalog")
dbutils.widgets.text("schema", "", "Schema")

In [ ]:
catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()

if not catalog or not schema:
    raise ValueError("Both `catalog` and `schema` widgets must be set.")

spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS ${catalog}.${schema}.dbspend360_pool_dbu_cost (
  instance_pool_id    STRING,
  cluster_id          STRING,
  usage_date          DATE,
  workspace_id        STRING,
  databricks_cost     DOUBLE,
  currency            STRING,
  sku_name            STRING,
  workspace_covered   BOOLEAN,
  created_at          TIMESTAMP,
  updated_at          TIMESTAMP
)
CLUSTER BY AUTO

In [ ]:
dbutils.notebook.exit(f"{catalog}.{schema}.dbspend360_pool_dbu_cost")